<a href="https://colab.research.google.com/github/MMujtabaX/agentic-ai-journey/blob/main/03_tool_calling_agent/tool_calling_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Tool-Calling Agent with LangChain

An agent decides **when to call tools**, reads their results, and decides what to do next. This one is a pun-loving weather assistant with:

- a **real weather tool** (Open-Meteo, free, no API key)
- a **context-aware tool** that reads the user's ID from runtime context
- **conversation memory** across turns (`InMemorySaver` + `thread_id`)
- a printed **trace** of every tool call

Model: a tool-calling model on **Groq** (Llama 3.3 70B or GPT-OSS), selected automatically from what your account can access.

> 🔑 Add `GROQ_API_KEY` to Colab Secrets.

In [ ]:
!pip install -qU langchain langchain-groq langgraph requests

In [ ]:
import requests
from dataclasses import dataclass
from google.colab import userdata
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver

from groq import Groq

# Groq retires models over time, so choose the first tool-calling model your account can access
available = {m.id for m in Groq(api_key=userdata.get("GROQ_API_KEY")).models.list().data}
for MODEL in ["llama-3.3-70b-versatile", "openai/gpt-oss-120b", "openai/gpt-oss-20b", "qwen/qwen3-32b"]:
    if MODEL in available:
        break
print("✅ Using model:", MODEL)

model = ChatGroq(model=MODEL, temperature=0, api_key=userdata.get("GROQ_API_KEY"))

## 1. Tools
A tool is a Python function with a docstring. The docstring is what the model reads to decide when to use it.

In [ ]:
@tool
def get_weather_for_location(city: str) -> str:
    """Get the current weather for a given city."""
    geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                       params={"name": city, "count": 1}, timeout=10).json()
    if not geo.get("results"):
        return f"Could not find a city called {city}."
    place = geo["results"][0]
    wx = requests.get("https://api.open-meteo.com/v1/forecast",
                      params={"latitude": place["latitude"], "longitude": place["longitude"],
                              "current": "temperature_2m,wind_speed_10m,relative_humidity_2m"},
                      timeout=10).json()["current"]
    return (f"{place['name']}, {place.get('country', '')}: {wx['temperature_2m']}°C, "
            f"humidity {wx['relative_humidity_2m']}%, wind {wx['wind_speed_10m']} km/h")

@dataclass
class Context:
    """Runtime context passed in with every request."""
    user_id: str

USER_CITIES = {"1": "Karachi", "2": "Lahore"}

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Look up the current user's city from their user ID."""
    return USER_CITIES.get(runtime.context.user_id, "Islamabad")

print(get_weather_for_location.invoke({"city": "Karachi"}))

## 2. Create the agent

In [ ]:
SYSTEM_PROMPT = """You are an expert weather forecaster who loves puns.

You have two tools:
- get_weather_for_location: get the current weather for a specific city
- get_user_location: find out where the user is

If the user asks about the weather "outside" or "here", first call get_user_location,
then get the weather for that city. Keep answers to 2-3 sentences."""

agent = create_agent(
    model=model,
    tools=[get_weather_for_location, get_user_location],
    system_prompt=SYSTEM_PROMPT,
    context_schema=Context,
    checkpointer=InMemorySaver(),
)

def show_trace(result):
    """Print every tool call and tool result, then the final answer."""
    for m in result["messages"]:
        for tc in getattr(m, "tool_calls", []) or []:
            print(f"  🔧 Tool call  : {tc['name']}({tc['args']})")
        if m.type == "tool":
            print(f"  📦 Tool result: {m.content}")
    print(f"\n🤖 {result['messages'][-1].content}")

## 3. Run it
The agent has to chain **two** tools: find the user's city, then fetch its weather.

In [ ]:
config = {"configurable": {"thread_id": "conversation-1"}}

result = agent.invoke({"messages": [{"role": "user", "content": "What's the weather like outside?"}]},
                      config=config, context=Context(user_id="1"))
show_trace(result)

## 4. Memory: a follow-up in the same thread
Because of the checkpointer and `thread_id`, the agent remembers the previous turn.

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "And how does that compare with Lahore?"}]},
                      config=config, context=Context(user_id="1"))
show_trace(result)

In [ ]:
# A different user (different context) gets a different location
result = agent.invoke({"messages": [{"role": "user", "content": "Is it windy where I am?"}]},
                      config={"configurable": {"thread_id": "conversation-2"}}, context=Context(user_id="2"))
show_trace(result)